# 04 - Feature Engineering

This notebook extracts interpretable ECG features from preprocessed 2-second segments and prepares a segment-level features table for the next modeling steps.


## Implemented Features

For each ECG segment and each lead, the current feature pipeline extracts:

- statistical features: `mean`, `std`, `min`, `max`, `amplitude`, `energy`, `skewness`, `kurtosis`
- morphological features: `area`, `abs_mean`, `rms`, `zero_crossing_rate`, `n_peaks`
- aggregated features across leads: `global_mean_*`, `global_std_*`, `global_min_*`, `global_max_*`

The notebook also creates the clinical utility label used later in modeling: `Atrial` vs `Non-Atrial`.


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from config import WFDB_RECORDS_DIR
from data_loading import load_record
from segmentation import segment_dataset, segment_record
from feature_extraction import build_features_dataframe, summarize_features


In [5]:
RECORD_LIMIT = 20
ATRIAL_CODES = {"164889003", "164890007"}


## Build Segment Dataset


In [6]:
hea_files = sorted(WFDB_RECORDS_DIR.rglob("*.hea"))
records = hea_files[:RECORD_LIMIT]

patient_segments, segments_array, segments_df, errors_df = segment_dataset(
    records=records,
    loader=load_record,
    lowcut=0.5,
    highcut=40.0,
    order=4,
    window_sec=2.0,
    step_sec=0.5,
)

print("Patients processed:", len(patient_segments))
print("Segments array shape:", segments_array.shape)
print("Segments dataframe shape:", segments_df.shape)
print("Errors:", len(errors_df))
segments_df.head()


Patients processed: 20
Segments array shape: (340, 1000, 12)
Segments dataframe shape: (340, 6)
Errors: 0


,patient_id,segment_id,label,segment_ref,start_sample,end_sample
0,JS00001,JS00001_seg_0000,"164889003,59118001,164934002",0,0,1000
1,JS00001,JS00001_seg_0001,"164889003,59118001,164934002",1,250,1250
2,JS00001,JS00001_seg_0002,"164889003,59118001,164934002",2,500,1500
3,JS00001,JS00001_seg_0003,"164889003,59118001,164934002",3,750,1750
4,JS00001,JS00001_seg_0004,"164889003,59118001,164934002",4,1000,2000


## Extract Features


In [7]:
example_record = segment_record(records[0].with_suffix(""), load_record)
leads = example_record["leads"]
fs = example_record["fs"]

features_df = build_features_dataframe(
    segments_array=segments_array,
    segments_df=segments_df,
    leads=leads,
    fs=fs,
    utility_codes=ATRIAL_CODES,
    positive_label="Atrial",
    negative_label="Non-Atrial",
    include_rr_features=False,
)

summary = summarize_features(features_df)
summary


{'n_rows': 340,
 'n_columns': 215,
 'n_numeric_features': 211,
 'missing_values': 0,
 'utility_label_counts': {'Non-Atrial': 272, 'Atrial': 68}}

In [8]:
features_df.head()


,patient_id,segment_id,label,utility_label,segment_ref,start_sample,end_sample,lead_I_mean,lead_I_std,lead_I_min,...,global_min_rms,global_max_rms,global_mean_zero_crossing_rate,global_std_zero_crossing_rate,global_min_zero_crossing_rate,global_max_zero_crossing_rate,global_mean_n_peaks,global_std_n_peaks,global_min_n_peaks,global_max_n_peaks
0,JS00001,JS00001_seg_0000,"164889003,59118001,164934002",Atrial,0,0,1000,-0.163769,1.209824,-4.846666,...,0.866558,1.490585,0.023357,0.007517,0.014014,0.038038,6.083333,1.114924,5.0,8.0
1,JS00001,JS00001_seg_0001,"164889003,59118001,164934002",Atrial,1,250,1250,0.153679,0.898510,-2.568021,...,0.701059,1.005323,0.024942,0.009700,0.013013,0.041041,6.166667,1.404358,4.0,8.0
2,JS00001,JS00001_seg_0002,"164889003,59118001,164934002",Atrial,2,500,1500,0.004842,0.836291,-2.321608,...,0.793182,1.265454,0.024358,0.009794,0.011011,0.039039,5.416667,1.381927,3.0,7.0
3,JS00001,JS00001_seg_0003,"164889003,59118001,164934002",Atrial,3,750,1750,0.042885,0.873478,-2.321608,...,0.809968,1.504668,0.026944,0.011269,0.009009,0.042042,5.833333,1.280191,3.0,7.0
4,JS00001,JS00001_seg_0004,"164889003,59118001,164934002",Atrial,4,1000,2000,0.022468,0.860830,-2.385116,...,0.786630,1.548732,0.028195,0.013632,0.009009,0.046046,5.916667,1.320248,3.0,7.0


## Label Check


In [9]:
features_df[["patient_id", "segment_id", "label", "utility_label"]].head(10)


,patient_id,segment_id,label,utility_label
0,JS00001,JS00001_seg_0000,"164889003,59118001,164934002",Atrial
1,JS00001,JS00001_seg_0001,"164889003,59118001,164934002",Atrial
2,JS00001,JS00001_seg_0002,"164889003,59118001,164934002",Atrial
3,JS00001,JS00001_seg_0003,"164889003,59118001,164934002",Atrial
4,JS00001,JS00001_seg_0004,"164889003,59118001,164934002",Atrial
5,JS00001,JS00001_seg_0005,"164889003,59118001,164934002",Atrial
6,JS00001,JS00001_seg_0006,"164889003,59118001,164934002",Atrial
7,JS00001,JS00001_seg_0007,"164889003,59118001,164934002",Atrial
8,JS00001,JS00001_seg_0008,"164889003,59118001,164934002",Atrial
9,JS00001,JS00001_seg_0009,"164889003,59118001,164934002",Atrial


In [10]:
features_df["utility_label"].value_counts(dropna=False)


utility_label
Non-Atrial    272
Atrial         68
Name: count, dtype: int64

## Feature Audit


In [11]:
numeric_features = features_df.select_dtypes(include="number")
zero_variance_features = numeric_features.columns[numeric_features.nunique() <= 1].tolist()
missing_counts = features_df.isna().sum()
feature_columns = [
    col for col in features_df.columns
    if col not in {"patient_id", "segment_id", "label", "utility_label", "segment_ref", "start_sample", "end_sample"}
]

print("Numeric columns:", numeric_features.shape[1])
print("Training feature columns:", len(feature_columns))
print("Zero-variance numeric features:", len(zero_variance_features))
print("Missing values total:", int(missing_counts.sum()))
zero_variance_features[:10]


Numeric columns: 211
Training feature columns: 208
Zero-variance numeric features: 0
Missing values total: 0


[]